In [ ]:
# Example 1: Construct a PriorityTree and insert a single stream.
from priority import PriorityTree

tree = PriorityTree()
tree.insert_stream(stream_id=1)

# A tree with one stream should yield that stream's id when iterated.
next_stream = next(iter(tree))
print(f"next stream id = {next_stream}")
assert next_stream == 1


In [ ]:
# Example 2: Insert multiple streams with explicit weights and iterate.
from collections import Counter
from priority import PriorityTree

tree = PriorityTree()
tree.insert_stream(stream_id=1, weight=16)
tree.insert_stream(stream_id=3, weight=32)
tree.insert_stream(stream_id=5, weight=64)

# Iterate a fixed number of times to see weighted distribution.
iterator = iter(tree)
counts = Counter(next(iterator) for _ in range(112))
print(f"counts after 112 iterations: {dict(counts)}")
assert set(counts.keys()) == {1, 3, 5}
# Higher-weighted streams should appear at least as often as lower-weighted ones.
assert counts[5] >= counts[3] >= counts[1]


In [ ]:
# Example 3: Block and unblock a stream; blocked streams are skipped by the iterator.
from priority import PriorityTree

tree = PriorityTree()
tree.insert_stream(stream_id=1)
tree.insert_stream(stream_id=3)

tree.block(stream_id=1)

iterator = iter(tree)
seen = {next(iterator) for _ in range(10)}
print(f"seen with stream 1 blocked: {seen}")
assert seen == {3}

tree.unblock(stream_id=1)
seen_after = {next(iterator) for _ in range(10)}
print(f"seen after unblocking 1: {seen_after}")
assert 1 in seen_after


In [ ]:
# Example 4: Reprioritize a stream and remove it from the tree.
from priority import PriorityTree

tree = PriorityTree()
tree.insert_stream(stream_id=1, weight=16)
tree.insert_stream(stream_id=3, weight=16)

# Bump stream 1's weight up and make stream 3 depend on stream 1.
tree.reprioritize(stream_id=1, weight=200)
tree.reprioritize(stream_id=3, depends_on=1, weight=16, exclusive=False)

# Now remove stream 1; stream 3 should still be reachable.
tree.remove_stream(stream_id=1)
remaining = next(iter(tree))
print(f"remaining stream after removing 1 = {remaining}")
assert remaining == 3


In [ ]:
# Example 5: Exception classes are exposed; duplicate insert raises DuplicateStreamError.
import priority
from priority import PriorityTree

# Verify the public exception hierarchy is importable.
for name in ("PriorityError", "DuplicateStreamError", "MissingStreamError",
             "TooManyStreamsError", "BadWeightError", "PseudoStreamError"):
    assert hasattr(priority, name), f"missing {name}"
    assert issubclass(getattr(priority, name), Exception)

tree = PriorityTree()
tree.insert_stream(stream_id=1)

raised = False
try:
    tree.insert_stream(stream_id=1)
except priority.DuplicateStreamError:
    raised = True

print(f"DuplicateStreamError raised on duplicate insert = {raised}")
assert raised
